# Santorini Colab Launch Recipe

This is the recommended notebook shape for longer Santorini training runs. Use a GPU runtime, keep checkpoints on Google Drive, and resume with replay examples loaded.

## 1. Mount Drive

In [7]:
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 2. Get the repo

In [8]:
%cd /content
!git clone https://github.com/Luminous9/alpha-zero-general.git
%cd /content/alpha-zero-general

/content
fatal: destination path 'alpha-zero-general' already exists and is not an empty directory.
/content/alpha-zero-general


In [ ]:
%cd /content/alpha-zero-general
!git pull

## 3. Install the Santorini training dependencies
Colab already includes PyTorch, so start with the lightweight dependencies.

In [3]:
!pip install coloredlogs tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.1 MB/s eta 0:00:00


Only use the full `requirements.txt` install if you need the older TensorFlow examples too; it pins old package versions that are not needed for Santorini.

## 4. Choose a Drive checkpoint folder

In [4]:
CHECKPOINT = "/content/drive/MyDrive/Santorini-AZ/checkpoints"

In [5]:
%env CHECKPOINT=/content/drive/MyDrive/Santorini-AZ/checkpoints

env: CHECKPOINT=/content/drive/MyDrive/Santorini-AZ/checkpoints


## 5. First long run
If you do not already have a checkpoint in Drive, start without `--load-model` or copy a local bootstrap checkpoint into `CHECKPOINT` first.

In [ ]:
!python main_santorini.py \
  --preset local \
  --checkpoint "$CHECKPOINT" \
  --num-iters 100 \
  --num-eps 80 \
  --num-mcts-sims 64 \
  --arena-compare 80 \
  --update-threshold 0.50 \
  --epochs 3 \
  --history-iters 5 \
  --seed 8 \
  --quiet

## 6. Resume a stopped run

This loads `best.pth.tar` by default. For that normal resume path, Santorini asks for `latest.examples` first, then falls back to model-adjacent examples, `best.pth.tar.examples`, and the newest checkpoint examples. An explicit `--examples-file` always wins.

In [ ]:
!python main_santorini.py \
  --preset local \
  --checkpoint "$CHECKPOINT" \
  --load-folder "$CHECKPOINT" \
  --load-model \
  --load-examples \
  --self-play-batch-size 64 \
  --batch-size 128 \
  --arena-batch-size 32 \
  --num-iters 5 \
  --num-eps 80 \
  --num-mcts-sims 64 \
  --arena-compare 80 \
  --update-threshold 0.50 \
  --epochs 3 \
  --history-iters 5 \
  --skip-first-self-play \
  --seed 9 \
  --quiet

2026-06-29 13:33:44 d0f9f1194ef3 __main__[6673] INFO Loading SantoriniGame...
2026-06-29 13:33:44 d0f9f1194ef3 __main__[6673] INFO Loading NNetWrapper...
2026-06-29 13:33:45 d0f9f1194ef3 __main__[6673] INFO Loading checkpoint "/content/drive/MyDrive/Santorini-AZ/checkpoints/best.pth.tar"...
2026-06-29 13:33:46 d0f9f1194ef3 __main__[6673] INFO Loading the Coach...
2026-06-29 13:33:46 d0f9f1194ef3 __main__[6673] INFO Loading 'trainExamples' from file...
2026-06-29 13:33:46 d0f9f1194ef3 Coach[6673] INFO Loading trainExamples from "/content/drive/MyDrive/Santorini-AZ/checkpoints/latest.examples"...
2026-06-29 13:33:53 d0f9f1194ef3 Coach[6673] INFO Loading done!
2026-06-29 13:33:53 d0f9f1194ef3 __main__[6673] INFO Config: preset=local iters=5 eps=80 sims=64 self_play_batch=32 arena=60 epochs=3 batch=64 checkpoint=/content/drive/MyDrive/Santorini-AZ/checkpoints
2026-06-29 13:33:53 d0f9f1194ef3 __main__[6673] INFO Starting the Santorini learning process
2026-06-29 13:33:53 d0f9f1194ef3 Coach[

Use `--examples-file filename.examples` when you want to force a specific replay file; relative paths are checked inside the load folder and from the launch directory. Use `--skip-first-self-play` only if the loaded examples were generated from the exact loaded model and you intentionally want to train immediately before collecting fresh games.